# Stage 1 — Connect to the Private GitHub Repository

## Purpose

Connect the Colab runtime to the existing private GitHub repository and use it as the project workspace.

## Repository

```text

swanksenia/health-psychology-rag-kb

```

## Authentication

The GitHub Personal Access Token is stored securely in Colab Secrets under:

```text
GITHUB_TOKEN
```

The token value is not written directly into the notebook.

## Expected output

The repository is cloned as a temporary working copy to:

```text
/content/health-psychology-rag-kb
```

The Homework 1 chunks will then be available at:

```text
/content/health-psychology-rag-kb/data/processed/chunks.jsonl
```

In [1]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess


GITHUB_USER = "swanksenia"
REPO_NAME = "health-psychology-rag-kb"

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
PROJECT_ROOT = Path("/content") / REPO_NAME


github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise ValueError(
        "GITHUB_TOKEN was not found in Colab Secrets."
    )


# Temporary helper for secure GitHub authentication.
askpass_path = Path("/content/git_askpass.sh")

askpass_path.write_text(
    """#!/bin/sh
case "$1" in
    *Username*) echo "x-access-token" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8",
)

askpass_path.chmod(0o700)


git_environment = os.environ.copy()
git_environment["GITHUB_TOKEN"] = github_token
git_environment["GIT_ASKPASS"] = str(askpass_path)
git_environment["GIT_TERMINAL_PROMPT"] = "0"


try:
    if (PROJECT_ROOT / ".git").exists():
        command = [
            "git",
            "-C",
            str(PROJECT_ROOT),
            "pull",
            "--ff-only",
        ]
        action = "updated"

    else:
        command = [
            "git",
            "clone",
            REPO_URL,
            str(PROJECT_ROOT),
        ]
        action = "cloned"

    subprocess.run(
        command,
        env=git_environment,
        check=True,
        capture_output=True,
        text=True,
    )

    print(f"✅ Repository successfully {action}.")
    print(f"Project root: {PROJECT_ROOT}")

except subprocess.CalledProcessError as error:
    error_message = error.stderr or "Unknown Git error"
    error_message = error_message.replace(github_token, "***")

    raise RuntimeError(
        f"GitHub connection failed:\n{error_message}"
    ) from error

finally:
    askpass_path.unlink(missing_ok=True)
    del github_token

✅ Repository successfully updated.
Project root: /content/health-psychology-rag-kb


# Stage 2 — Load and Inspect Homework 1 Chunks

## Purpose

Load the `chunks.jsonl` file created in Homework 1 and inspect the input data before generating embeddings.

## Input

```text
data/processed/chunks.jsonl

In [2]:
import json


INPUT_CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks.jsonl"
)


def load_jsonl(path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                records.append(json.loads(line))

    return records


chunks = load_jsonl(INPUT_CHUNKS_PATH)

print(f"Loaded chunks: {len(chunks)}")
print(f"Input path: {INPUT_CHUNKS_PATH}")
print()
print("First chunk:")
print(json.dumps(chunks[0], indent=2, ensure_ascii=False))

Loaded chunks: 474
Input path: /content/health-psychology-rag-kb/data/processed/chunks.jsonl

First chunk:
{
  "chunk_id": "health_psychology_course_syllabus__0000",
  "document_id": "health_psychology_course_syllabus",
  "source_file": "data/raw/course_syllabus.pdf",
  "chunk_index": 0,
  "section": "Document overview",
  "text": "Syllabus for Introduction to Health Psychology Credits: 3 PSYC 1111 Instructor Contact Information: You can always send your instructor a private message through the Brightspace Messaging system, accessible via the envelope (Messages) icon in the top navigation bar. Once logged into your course, click your instructor’s profile page to see all the ways you can communicate with them, including their email address. Course Description Health psychology focuses on the dynamic interaction between biological, social, and psychological factors that influence physical health and illness, aiming to promote overall well-being and prevent diseases. This course is design

# Stage 3 — Demo 1: Embeddings and Similarity

## Purpose

Demonstrate how text is converted into embedding vectors and compare the semantic similarity of several Health Psychology examples.

## Model

```text
sentence-transformers/all-MiniLM-L6-v2
```

## What this stage demonstrates

- each text is converted into a numeric embedding vector;
- semantically similar texts should have higher cosine similarity;
- semantically similar texts should have lower Euclidean distance;
- chunk and query embeddings must be created with the same embedding model.

## Important

This stage reproduces the first Lesson 4 demo.

It is a small demonstration only and does not yet create embeddings for all Homework 1 chunks.

In [3]:
!pip install -q \
    sentence-transformers==3.0.1 \
    faiss-cpu==1.8.0.post1 \
    numpy==1.26.4

In [4]:
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer


print("NumPy version:", np.__version__)
print("FAISS version:", getattr(faiss, "__version__", "available"))
print("SentenceTransformer import: OK")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


NumPy version: 1.26.4
FAISS version: 1.8.0
SentenceTransformer import: OK


In [5]:
import numpy as np

from sentence_transformers import SentenceTransformer


MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


def cosine_similarity(vector_a, vector_b):
    return np.dot(vector_a, vector_b) / (
        np.linalg.norm(vector_a)
        * np.linalg.norm(vector_b)
    )


def euclidean_distance(vector_a, vector_b):
    return np.linalg.norm(vector_a - vector_b)


print("=" * 80)
print("LESSON 4 DEMO 1: EMBEDDINGS AND SIMILARITY")
print("=" * 80)
print(f"Embedding model: {MODEL_NAME}")
print("Loading model...")


model = SentenceTransformer(MODEL_NAME)


texts = [
    "biopsychosocial model of health",
    "biological psychological and social factors in illness",
    "employee vacation policy",
    "behaviour change interventions",
]


embeddings = model.encode(
    texts,
    convert_to_numpy=True,
)


print()
print(f"Number of texts: {len(texts)}")
print(f"Embedding shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")


print()
print("-" * 80)
print("Example embedding vector")
print("-" * 80)
print(f"Text: {texts[0]}")
print("First 10 vector values:")
print(np.round(embeddings[0][:10], 4))


pairs_to_compare = [
    (0, 1),
    (0, 2),
    (0, 3),
]


print()
print("-" * 80)
print("Similarity comparison")
print("-" * 80)


for first_index, second_index in pairs_to_compare:
    first_text = texts[first_index]
    second_text = texts[second_index]

    first_embedding = embeddings[first_index]
    second_embedding = embeddings[second_index]

    similarity = cosine_similarity(
        first_embedding,
        second_embedding,
    )

    distance = euclidean_distance(
        first_embedding,
        second_embedding,
    )

    print()
    print(f"Text A: {first_text}")
    print(f"Text B: {second_text}")
    print(f"Cosine similarity: {similarity:.4f}")
    print(f"Euclidean distance: {distance:.4f}")


print()
print("-" * 80)
print("Key takeaway")
print("-" * 80)
print(
    "Semantically similar texts should have higher cosine "
    "similarity and lower Euclidean distance."
)
print(
    "Chunks and user queries must be encoded with the same model."
)

LESSON 4 DEMO 1: EMBEDDINGS AND SIMILARITY
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Number of texts: 4
Embedding shape: (4, 384)
Embedding dimension: 384

--------------------------------------------------------------------------------
Example embedding vector
--------------------------------------------------------------------------------
Text: biopsychosocial model of health
First 10 vector values:
[ 0.0427  0.0032 -0.0576  0.0385 -0.0745  0.0115  0.0354  0.1066 -0.0158
 -0.0014]

--------------------------------------------------------------------------------
Similarity comparison
--------------------------------------------------------------------------------

Text A: biopsychosocial model of health
Text B: biological psychological and social factors in illness
Cosine similarity: 0.5831
Euclidean distance: 0.9131

Text A: biopsychosocial model of health
Text B: employee vacation policy
Cosine similarity: -0.0009
Euclidean distance: 1.4149

Text A: biopsychosocial model of health
Text B: behaviour change interventions
Cosine similarity: 0.3081
Euclidean distance: 

# Stage 4 — Demo 2: Build the FAISS Index

## Purpose

Create embeddings for all Homework 1 chunks and store the normalized vectors in a local FAISS index.

## Process

```text
chunks.jsonl
→ chunk texts
→ embeddings
→ L2 normalization
→ FAISS IndexFlatIP
```

## Model

```text
sentence-transformers/all-MiniLM-L6-v2
```

The same embedding model will later be used for user queries.

## Storage approach

FAISS stores the embedding vectors.

Chunk text and metadata remain in a separate JSONL file. The order of chunks in the JSONL file must match the order of vectors in the FAISS index.

## Expected outputs

```text
data/processed/chunks_for_retrieval.jsonl
index/embeddings.npy
index/faiss.index
```

In [6]:
import json

import faiss
import numpy as np

from sentence_transformers import SentenceTransformer


MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

INPUT_CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks.jsonl"
)

OUTPUT_CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks_for_retrieval.jsonl"
)

OUTPUT_EMBEDDINGS_PATH = (
    PROJECT_ROOT
    / "index"
    / "embeddings.npy"
)

OUTPUT_INDEX_PATH = (
    PROJECT_ROOT
    / "index"
    / "faiss.index"
)


def load_jsonl(path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                records.append(json.loads(line))

    return records


def save_jsonl(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(
                json.dumps(record, ensure_ascii=False)
                + "\n"
            )


def normalize_embeddings(embeddings):
    embeddings = np.asarray(
        embeddings,
        dtype=np.float32,
    )

    embeddings = np.ascontiguousarray(embeddings)

    faiss.normalize_L2(embeddings)

    return embeddings


print("=" * 80)
print("LESSON 4 DEMO 2: BUILD FAISS INDEX")
print("=" * 80)

print(f"Input chunks: {INPUT_CHUNKS_PATH}")
print(f"Embedding model: {MODEL_NAME}")
print()


chunks = load_jsonl(INPUT_CHUNKS_PATH)

texts = [
    chunk["text"]
    for chunk in chunks
]

print(f"Loaded chunks: {len(chunks)}")
print("Loading embedding model...")


model = SentenceTransformer(MODEL_NAME)


print("Creating chunk embeddings...")

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True,
)


print(f"Embeddings shape: {embeddings.shape}")

normalized_embeddings = normalize_embeddings(
    embeddings
)

embedding_dimension = normalized_embeddings.shape[1]


print("Building FAISS index...")

index = faiss.IndexFlatIP(
    embedding_dimension
)

index.add(
    normalized_embeddings
)


OUTPUT_EMBEDDINGS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_INDEX_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


save_jsonl(
    chunks,
    OUTPUT_CHUNKS_PATH,
)

np.save(
    OUTPUT_EMBEDDINGS_PATH,
    normalized_embeddings,
)

faiss.write_index(
    index,
    str(OUTPUT_INDEX_PATH),
)


print()
print("-" * 80)
print("FAISS index created")
print("-" * 80)

print(f"Index type: {type(index).__name__}")
print(f"Vectors in index: {index.ntotal}")
print(f"Vector dimension: {index.d}")
print()

print(f"Chunks saved to: {OUTPUT_CHUNKS_PATH}")
print(f"Embeddings saved to: {OUTPUT_EMBEDDINGS_PATH}")
print(f"FAISS index saved to: {OUTPUT_INDEX_PATH}")

LESSON 4 DEMO 2: BUILD FAISS INDEX
Input chunks: /content/health-psychology-rag-kb/data/processed/chunks.jsonl
Embedding model: sentence-transformers/all-MiniLM-L6-v2

Loaded chunks: 474
Loading embedding model...
Creating chunk embeddings...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embeddings shape: (474, 384)
Building FAISS index...

--------------------------------------------------------------------------------
FAISS index created
--------------------------------------------------------------------------------
Index type: IndexFlatIP
Vectors in index: 474
Vector dimension: 384

Chunks saved to: /content/health-psychology-rag-kb/data/processed/chunks_for_retrieval.jsonl
Embeddings saved to: /content/health-psychology-rag-kb/index/embeddings.npy
FAISS index saved to: /content/health-psychology-rag-kb/index/faiss.index


# Stage 5 — Demo 3: Semantic Search

## Purpose

Implement baseline top-k semantic search over the FAISS index created in Stage 4.

## Retrieval pipeline

```text
user query
→ query embedding
→ L2 normalization
→ FAISS search
→ top-k vector positions
→ corresponding chunks and metadata
```

## Returned information

For each retrieved result, display:

- rank;
- similarity score;
- `chunk_id`;
- `document_id`;
- `source_file`;
- `section`;
- text preview.

## Important

The user query is encoded with the same model that was used to create the chunk embeddings.

This is baseline semantic retrieval without metadata filtering, hybrid search, reranking, or query rewriting.

In [7]:
import json

import faiss
import numpy as np

from sentence_transformers import SentenceTransformer


MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

CHUNKS_FOR_RETRIEVAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks_for_retrieval.jsonl"
)

FAISS_INDEX_PATH = (
    PROJECT_ROOT
    / "index"
    / "faiss.index"
)


def load_jsonl(path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                records.append(json.loads(line))

    return records


def search(
    query,
    model,
    index,
    chunks,
    top_k=3,
):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32,
    )

    query_embedding = np.ascontiguousarray(
        query_embedding
    )

    faiss.normalize_L2(query_embedding)

    scores, positions = index.search(
        query_embedding,
        top_k,
    )

    results = []

    for score, position in zip(
        scores[0],
        positions[0],
    ):
        chunk = chunks[position]

        results.append(
            {
                "chunk_id": chunk.get("chunk_id"),
                "score": float(score),
                "text": chunk.get("text", ""),
                "metadata": {
                    "document_id": chunk.get("document_id"),
                    "source_file": chunk.get("source_file"),
                    "chunk_index": chunk.get("chunk_index"),
                    "section": chunk.get("section"),
                },
            }
        )

    return results


print("=" * 80)
print("LESSON 4 DEMO 3: SEMANTIC SEARCH")
print("=" * 80)


chunks_for_retrieval = load_jsonl(
    CHUNKS_FOR_RETRIEVAL_PATH
)

index = faiss.read_index(
    str(FAISS_INDEX_PATH)
)

model = SentenceTransformer(
    MODEL_NAME
)


query = "What is the biopsychosocial model of health?"
top_k = 3


print(f"Query: {query}")
print(f"Top-k: {top_k}")
print()


results = search(
    query=query,
    model=model,
    index=index,
    chunks=chunks_for_retrieval,
    top_k=top_k,
)


print("-" * 80)
print("Retrieved chunks")
print("-" * 80)


for rank, result in enumerate(
    results,
    start=1,
):
    metadata = result["metadata"]

    text_preview = result["text"][:500].replace(
        "\n",
        " ",
    )

    print()
    print(f"Rank: {rank}")
    print(f"Score: {result['score']:.4f}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(
        f"Document ID: "
        f"{metadata.get('document_id')}"
    )
    print(
        f"Source file: "
        f"{metadata.get('source_file')}"
    )
    print(
        f"Section: "
        f"{metadata.get('section')}"
    )
    print(f"Text preview: {text_preview}")
    print("-" * 80)

LESSON 4 DEMO 3: SEMANTIC SEARCH
Query: What is the biopsychosocial model of health?
Top-k: 3

--------------------------------------------------------------------------------
Retrieved chunks
--------------------------------------------------------------------------------

Rank: 1
Score: 0.7959
Chunk ID: ogden_2019_health_psychology__0013
Document ID: ogden_2019_health_psychology
Source file: data/raw/ogden_2019_health_psychology.pdf
Section: 1.The Biopsychosocial Model
Text preview: smoking), pressures to change behavior (e.g. peer group expectations, parental pressure), social values on health (e.g. whether health was regarded as a good or a bad thing), social class, the environment, and ethnicity. #### Fig 1 The biopsychosocial model of health and illness (after Engel 1977, 1980) ![Fig 1 The biopsychosocial model of health and illness (after Engel 1977, 1980)](assets/ogden_p7_figure_1.png) Figure description: The biopsychosocial model explains health and illness through t
---------

# Stage 6 — Test Queries and Retrieval Evaluation

## Purpose

Evaluate the semantic retrieval layer using representative questions from the Health Psychology knowledge base.

## Evaluation process

For each query:

1. create a query embedding with the same model;
2. retrieve the top 3 chunks from the FAISS index;
3. inspect the scores, text previews, and metadata;
4. evaluate whether the retrieved results are relevant.

## Test coverage

The queries cover:

- the biopsychosocial model;
- the COM-B model;
- the Behaviour Change Wheel;
- the 3P model of disease;
- stress and health;
- course structure and required readings.

## Output

The retrieval results will be saved to:

```text
outputs/retrieval_examples.md
```

In [8]:
TEST_QUERIES = [
    "What is the biopsychosocial model of health?",
    "What are the components of the COM-B model?",
    "How does the Behaviour Change Wheel support intervention design?",
    "What are the predisposing, precipitating, and perpetuating factors in the 3P model?",
    "How can stress affect physical health?",
    "What topics are covered in the Health Psychology course?",
]


for query_number, query in enumerate(
    TEST_QUERIES,
    start=1,
):
    results = search(
        query=query,
        model=model,
        index=index,
        chunks=chunks_for_retrieval,
        top_k=3,
    )

    print("=" * 80)
    print(f"Query {query_number}: {query}")
    print("=" * 80)

    for rank, result in enumerate(
        results,
        start=1,
    ):
        metadata = result["metadata"]

        text_preview = (
            result["text"][:350]
            .replace("\n", " ")
        )

        print()
        print(
            f"Top-{rank}: "
            f"{result['chunk_id']} | "
            f"score: {result['score']:.4f}"
        )
        print(
            f"Source: "
            f"{metadata.get('source_file')}"
        )
        print(
            f"Section: "
            f"{metadata.get('section')}"
        )
        print(
            f"Text preview: "
            f"{text_preview}"
        )

    print()

Query 1: What is the biopsychosocial model of health?

Top-1: ogden_2019_health_psychology__0013 | score: 0.7959
Source: data/raw/ogden_2019_health_psychology.pdf
Section: 1.The Biopsychosocial Model
Text preview: smoking), pressures to change behavior (e.g. peer group expectations, parental pressure), social values on health (e.g. whether health was regarded as a good or a bad thing), social class, the environment, and ethnicity. #### Fig 1 The biopsychosocial model of health and illness (after Engel 1977, 1980) ![Fig 1 The biopsychosocial model of health a

Top-2: wright_2019_3p_disease_model__0010 | score: 0.7779
Source: data/raw/wright_2019_3p_disease_model.html
Section: Introduction
Text preview: and disease are conceptualized or managed today ([Kontos, 2011](https://pmc.ncbi.nlm.nih.gov/articles/PMC6879427/#B55)). This could be, in part, because the biopsychosocial model lacks a framework for understanding *how* biological, psychological, and socio-environmental factors may contr

# Stage 7 — Save Retrieval Examples

## Purpose

Save the six test queries, their top-3 retrieved chunks, and a short relevance evaluation.

## Output

```text
outputs/retrieval_examples.md

In [12]:
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

RETRIEVAL_EXAMPLES_PATH = (
    OUTPUTS_DIR
    / "retrieval_examples.md"
)


QUERY_COMMENTS = {
    "What is the biopsychosocial model of health?": (
        "Relevant. All three retrieved chunks discuss the "
        "biopsychosocial model. Top-1 provides the most direct "
        "explanation, while Top-2 and Top-3 add context about "
        "its structure, development, and limitations."
    ),
    "What are the components of the COM-B model?": (
        "Partially relevant. All retrieved chunks are related "
        "to COM-B, but the previews do not provide a single "
        "clear enumeration of capability, opportunity, "
        "motivation, and behaviour."
    ),
    "How does the Behaviour Change Wheel support intervention design?": (
        "Relevant. The retrieved chunks explain how the "
        "Behaviour Change Wheel links behaviour analysis, "
        "intervention functions, and policy categories."
    ),
    "What are the predisposing, precipitating, and perpetuating factors in the 3P model?": (
        "Relevant. Top-1 directly identifies the three factors, "
        "while Top-2 and Top-3 provide additional application "
        "and research context."
    ),
    "How can stress affect physical health?": (
        "Relevant. The retrieved chunks describe direct "
        "physiological pathways, indirect behavioural pathways, "
        "and links between stress and physical illness."
    ),
    "What topics are covered in the Health Psychology course?": (
        "Partially relevant. The syllabus appears in Top-2, "
        "but Top-1 and Top-3 provide broad descriptions of "
        "health psychology rather than a structured list of "
        "course topics."
    ),
}


markdown_lines = [
    "# Semantic Retrieval Examples",
    "",
    f"**Embedding model:** `{MODEL_NAME}`",
    "",
    "**Vector index:** `FAISS IndexFlatIP`",
    "",
    "**Top-k:** `3`",
    "",
]


for query_number, query in enumerate(
    TEST_QUERIES,
    start=1,
):
    results = search(
        query=query,
        model=model,
        index=index,
        chunks=chunks_for_retrieval,
        top_k=3,
    )

    markdown_lines.append(
        f"## Query {query_number}"
    )
    markdown_lines.append("")
    markdown_lines.append(
        f"**Query:** {query}"
    )
    markdown_lines.append("")

    for rank, result in enumerate(
        results,
        start=1,
    ):
        metadata = result["metadata"]

        text_preview = (
            result["text"][:350]
            .replace("\n", " ")
            .strip()
        )

        markdown_lines.append(
            f"### Top-{rank}"
        )
        markdown_lines.append("")

        markdown_lines.append(
            f"- **Chunk ID:** `{result['chunk_id']}`"
        )

        markdown_lines.append(
            f"- **Score:** `{result['score']:.4f}`"
        )

        markdown_lines.append(
            f"- **Document ID:** "
            f"`{metadata.get('document_id')}`"
        )

        markdown_lines.append(
            f"- **Source:** "
            f"`{metadata.get('source_file')}`"
        )

        markdown_lines.append(
            f"- **Section:** "
            f"`{metadata.get('section')}`"
        )

        markdown_lines.append(
            f"- **Text preview:** {text_preview}"
        )

        markdown_lines.append("")

    markdown_lines.append(
        f"**Comment:** {QUERY_COMMENTS[query]}"
    )

    markdown_lines.append("")
    markdown_lines.append("---")
    markdown_lines.append("")


markdown_lines.extend(
    [
        "# Overall Conclusion",
        "",
        (
            "The baseline semantic retrieval layer performs well "
            "for focused conceptual questions about named models "
            "and clearly defined health psychology topics."
        ),
        "",
        (
            "Retrieval is weaker for broader navigational "
            "questions and questions that require a precise list "
            "from a specific source. The current system uses "
            "semantic vector similarity only and does not apply "
            "metadata filtering, hybrid retrieval, reranking, "
            "or query rewriting."
        ),
        "",
    ]
)


RETRIEVAL_EXAMPLES_PATH.write_text(
    "\n".join(markdown_lines),
    encoding="utf-8",
)


print("✅ Retrieval examples saved.")
print("Output path:", RETRIEVAL_EXAMPLES_PATH)

✅ Retrieval examples saved.
Output path: /content/health-psychology-rag-kb/outputs/retrieval_examples.md


## Generated Outputs

The required retrieval artifacts were generated and saved in the project repository.

In [13]:
GENERATED_FILES = [
    OUTPUT_CHUNKS_PATH,
    OUTPUT_EMBEDDINGS_PATH,
    OUTPUT_INDEX_PATH,
    RETRIEVAL_EXAMPLES_PATH,
]

for file_path in GENERATED_FILES:
    print(
        f"{'✅' if file_path.exists() else '❌'} "
        f"{file_path.relative_to(PROJECT_ROOT)}"
    )

✅ data/processed/chunks_for_retrieval.jsonl
✅ index/embeddings.npy
✅ index/faiss.index
✅ outputs/retrieval_examples.md


# Stage 8 — Conclusion

## Result

A baseline semantic retrieval layer was created for the PSYC 1111 Health Psychology knowledge base.

The system:

- loaded 474 chunks created in Homework 1;
- generated embeddings with `sentence-transformers/all-MiniLM-L6-v2`;
- normalized the embeddings for cosine-like similarity search;
- stored 474 vectors in a FAISS `IndexFlatIP`;
- encoded user queries with the same embedding model;
- returned the top-3 chunks with similarity scores, text previews, and metadata;
- evaluated retrieval quality using six test queries.

## Retrieval quality

The retrieval layer performs well for focused questions about clearly named concepts, including:

- the biopsychosocial model;
- the Behaviour Change Wheel;
- the 3P model;
- stress and physical health.

Retrieval is weaker for:

- broad navigational questions;
- questions requiring a precise list;
- questions where the relevant information is distributed across several chunks.

## Limitations

The current implementation uses semantic vector similarity only.

It does not include:

- metadata filtering;
- hybrid retrieval;
- reranking;
- query rewriting;
- LLM-generated answers.

These improvements can be added in future iterations.